# Box Dimension Optimization

## 1. Context

At **TikiNOW**, warehouse operations use a fixed collection of carton box sizes — ranging from small (e.g. 5×5×5 cm) to large (e.g. 40×40×40 cm) — to pack and ship customer orders. This collection has been in use for years without systematic re-evaluation.

**The core question**: Given our current product portfolio, is this collection of box dimensions still optimal — or can we find a better set of sizes that reduces cost and improves space utilization?

## 2. Scope

- **Optimize the dimensions** of a fixed number of box types (a "Collection") used across the entire warehouse.
- **Evaluate** each candidate Collection by simulating the packing of historical orders using the existing `algorithm_BPS` (Box Packing Solution).
- **Compare** two Genetic Algorithm strategies:
  - **Naive GA** (`run()`): standard tournament selection + single-point crossover.
  - **Region-Informed GA** (`run_by_region()`): a novel hybrid approach leveraging per-region fitness decomposition and gene pool recombination.
- Results are intended for **paper publication** comparing convergence and final fitness of both approaches.

## 3. Objectives

We minimize a **weighted combination** of three objective functions, computed only over **fittable** orders (f1, f2) and over all orders (f3):

### $f_1$ — Cost Minimization (weight: 0.60)

$$f_1 = \frac{1}{N_{\text{fittable}}} \sum_{i=1}^{N_{\text{fittable}}} \left( C_{\text{box},i} + C_{\text{bubble},i} \right)$$

- $C_{\text{box},i} = 2(l_i w_i + l_i h_i + w_i h_i) \times \text{unit\_cost}$ — box cost proportional to surface area
- $C_{\text{bubble},i} = \frac{61000}{35 \times 90 \times 100 \times t} \times (V_{\text{box},i} - V_{\text{items},i}) \times \text{filling\_rate}$ — bubble wrap cost for unused space

### $f_2$ — Utilization Optimization (weight: 0.35)

$$f_2 = \frac{1}{N_{\text{fittable}}} \sum_{i=1}^{N_{\text{fittable}}} \left( \frac{V_{\text{items},i}}{V_{\text{box},i}} - u^* \right)^2$$

MSE of box utilization vs. target $u^* = 0.9$. Penalizes both over-packing and under-packing equally.

### $f_3$ — Unfittable Ratio (weight: 0.05)

$$f_3 = \frac{N_{\text{unfittable}}}{N_{\text{total}}}$$

### Combined Fitness $F$ (lower is better)

$$F = w_1 \cdot \frac{f_1}{b_1} + w_2 \cdot \frac{f_2}{b_2} + w_3 \cdot \left[e^{k(f_3/b_3 - 1)} - 1\right]$$

where $b_1, b_2, b_3$ are baseline values from the current Collection, and $k$ controls the exponential penalty sensitivity on unfittable ratio.

## 4. Limitations & Conditions

- **Box dimensions are integers** (in cm), clamped to $[\text{dim\_min},\ \text{dim\_max}]$ (default: 5–60 cm).
- **Shopee carton box length limit** applies: [Shopee Regulation](https://banhang.shopee.vn/edu/article/3639).
- **Unfittable orders are structurally expected** — some orders have too many items or volumes exceeding all boxes. A natural baseline of unfittable orders exists regardless of Collection; the exponential penalty targets only Collections significantly worse than this baseline.
- **Evaluation cost** is high — each fitness evaluation runs `algorithm_BPS` on every sampled order. This is mitigated by stratified proportional sampling + parallel processing (multiprocessing across buckets, threading within).

## 5. Approach

### 5.1 Data Pipeline
1. **Input**: Historical order data — `[order_code, item, unit, length, width, height]`.
2. **Reinforcement adjustment**: Item dimensions increased by $2 \times t_\text{reinforcement}$ per axis before packing.
3. **Stratified sampling** (`Sample`): Proportional sample preserving distributions of total volume, max dimension, and total units — validated by KS test + valuation comparison.

### 5.2 Genetic Algorithm
- **Encoding**: Each individual = a Collection of $N$ boxes, each with integer (L, W, H) sorted descending; boxes sorted by volume ascending.
- **Operators**: Tournament selection · single-point crossover · Gaussian mutation (rounded to int) · elitism · random immigrants.
- **Two run modes**:
  - `run()` — standard GA loop.
  - `run_by_region()` — builds a **selection pool** via per-region tournaments, identifies top-contributing boxes per region, assembles children from the pool (controlled by `pool_ratio`), fills the rest with traditional crossover.

### 5.3 Region Decomposition
Orders are grouped into **regions** defined by `(volume_bin, length_bin, unit_bin)` — the same quantile bins from stratified sampling (up to $5^3 = 125$ regions). Per-region fitness and box usage tracking enable the region-informed crossover to identify *which boxes serve which types of orders best*.

> Full methodology details: see `METHODOLOGY.md`.

In [12]:
from module.Module import *
# pyright: ignore[reportUndefinedVariable]

In [26]:
# ── Collection ──
NUMBER_OF_BOXES = 10  # how many box types in a Collection

# ── Valuation — physical constants ──
UTILIZATION_OPTIMAL = 0.8  # ideal utilization ratio (80%)
REINFORCEMENT_THICKNESS = 0.5  # cm per side
BUBBLE_THICKNESS = 1            # cm
BUBBLE_FILLING_RATE = 0.5

# ── Valuation — baselines for normalization (calibrate from current Collection) ──
OBJECTIVE_FUNCTION_1_BASELINE = 5000       # cost baseline
OBJECTIVE_FUNCTION_2_BASELINE = 0.1        # utilization deviation baseline
OBJECTIVE_FUNCTION_3_BASELINE = 0.05       # unfittable ratio baseline

# ── Valuation — objective function weights & penalty ──
W1 = 0.60   # cost weight
W2 = 0.35   # utilization weight
W3 = 0.05   # unfittable ratio weight
K  = 3.0    # exponential penalty sensitivity for f3

# ── Sample — stratified sampling ──
SAMPLE_SIZE = 100000
NUMBER_OF_BUCKETS_PER_ASPECT = 5  # quantile bins per stratification feature

# ── Sample — KS test thresholds ──
P_VALUE_THRESHOLD = 0.05
KS_STAT_THRESHOLD = 0.1

# ── Sample — valuation test ──
VALUATION_TEST_TOLERANCE = 0.1  # max relative difference between sample and full valuation

# ── Parallel processing ──
NUMBER_OF_BUCKETS = 5  # multiprocessing buckets for Valuation

# ── GA — core ──
POPULATION_SIZE = 50
GENERATIONS = 100
MUTATION_RATE = 0.1
CROSSOVER_RATE = 0.8
TOURNAMENT_SIZE = 3
ELITISM_COUNT = 2
IMMIGRANT_COUNT = 3
DIM_MIN = 5    # cm
DIM_MAX = 60   # cm
MUTATION_SIGMA = 3.0  # Gaussian perturbation std dev (cm)

# ── GA — region-informed (run_by_region only) ──
REGION_TOURNAMENT_SIZE = 20
REGION_TOURNAMENT_ROUNDS = 3
TOP_BOXES_PER_REGION = 3
POOL_RATIO = 0.5  # fraction of children from region pool vs. traditional crossover

In [15]:
# Process seperately for External and TikiCorp
data_order_external_1 = pd.read_csv('data/data_order_external_1.csv')
data_order_external_2 = pd.read_csv('data/data_order_external_2.csv')
# join the two external datasets for a comprehensive view
data_order_external = pd.concat([data_order_external_1, data_order_external_2], ignore_index=True)
data_order_external.head()
print(f"External dataset shape: {data_order_external.shape}")

External dataset shape: (9977989, 10)


In [16]:
data_box_external = pd.read_csv('data/data_box_external.csv')
data_box_external.head()

,box_code,length,width,height
0,E00,13.0,7.0,4.0
1,E01,17.0,11.0,4.0
2,EC2,19.5,10.5,10.5
3,E03,22.0,17.0,5.0
4,EC0A,25.0,33.0,4.0


In [24]:
current_collection_external = Collection(
    number_of_boxes=data_box_external.nunique()['box_code'],
    boxes_dimensions=data_box_external[['length', 'width', 'height']].values.tolist()
)

In [28]:
sample = Sample(
    sample_size=SAMPLE_SIZE,
    number_of_buckets_per_aspect=NUMBER_OF_BUCKETS_PER_ASPECT,
    data_order=data_order_external
)

print(sample.ks_test(P_VALUE_THRESHOLD, KS_STAT_THRESHOLD))
print(sample.valuation_test(filename= 'sample_test.csv', tolerance=VALUATION_TEST_TOLERANCE, collection=current_collection_external))

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x106336930>>
Traceback (most recent call last):
  File "/Users/lap-01102/Library/Python/3.12/lib/python/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


KeyboardInterrupt: 